# latex-task2 · De novo drug-design tables → LaTeX

Vina, PoseCheck / PoseBusters, strain, interaction, and reference-ligand similarity from `results/reports/results_drug_design.html`.

## Setup

In [1]:
import sys
from pathlib import Path
def _find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "voxbind" / "dataset").is_dir():
            return c
    fb = Path("/home/shpark/prj-denovo/VoxBind")
    if (fb / "voxbind" / "dataset").is_dir():
        return fb
    raise FileNotFoundError("repo root (voxbind/dataset) not found from cwd")
_REPO = _find_repo_root()
sys.path.insert(0, str(_REPO / "notebook" / "results"))
from bs4 import BeautifulSoup, Tag
from latex_common import *   # shared HTML->LaTeX helpers + generic table_to_latex

# ── task2 config (de novo drug design report) ──
DRUG_DESIGN_HTML_PATH = Path("results/reports/results_drug_design.html")
REFERENCE_SIMILARITY_HTML_PATH = Path("notebook/html/260910/fig-ref-ligand-similarity/reference_similarity_table.html")
REFERENCE_SIMILARITY_EXCLUDE_TAGS = True
REFERENCE_SIMILARITY_TWO_DECIMALS = False
KEEP_BADGES = True
RESIZE_WIDE = True


## Converters (drug design)

In [2]:
REFERENCE_SIMILARITY_CAPTION = (
    r"\textbf{Reference-ligand similarity} on the CrossDocked benchmark. "
    r"Metrics are averaged over pockets."
)
REFERENCE_SIMILARITY_LABEL = "tab:result-drug-reference-similarity"


def reference_similarity_to_latex(table: Tag, wrap: bool = False) -> str:
    """Transpose reference-ligand similarity: metrics as rows, methods as columns.

    The source is build_reference_similarity.py's fragment, not results.html. That
    script cut the metric set from the seven fingerprints of the 260820 table down to
    the two this literature actually reports — ECFP4/Morgan (radius 2, 2048 bit)
    Tanimoto, and the Bemis-Murcko scaffold match rate. MACCS / AtomPair / RDKit /
    Dice come back under --full and 3D shape under --with-3d, so the groups are read
    off the fragment's two-row header rather than hardcoded: changing the fingerprint
    set again does not touch this function.

    Layout: two stub columns, a metric group name and its statistic. A group with
    several statistics (ECFP4 -> mean / median) gets a \multirow name; a
    single-statistic group (Scaffold match) spans both stub columns with its name
    broken over two lines. \shortstack is plain LaTeX, so this needs only booktabs
    and multirow — no makecell.

    wrap=True emits the 260820 layout instead: a \resizebox'd wraptable for sitting
    beside the body text (needs wrapfig), same rows either way.

    Both outputs are byte-identical to the .tex files the builder writes
    (reference_similarity.tex and _wrap.tex); the two entry points exist so a
    regression in either is a one-line diff to catch.
    """
    def plain(cell) -> str:
        text = cell_to_latex(cell, keep_badges=not REFERENCE_SIMILARITY_EXCLUDE_TAGS)
        bold_match = re.fullmatch(r"\\textbf\{([^{}]+)\}", text)
        return bold_match.group(1) if bold_match else text

    header_rows = table.select("thead > tr")
    if not header_rows:
        raise ValueError("reference-similarity 표에 <thead>가 없습니다.")
    top = header_rows[0].find_all("th", recursive=False)[1:]   # 첫 칸은 Method stub
    subs = header_rows[1].find_all("th", recursive=False) if len(header_rows) > 1 else []

    groups, sub_index = [], 0
    for cell in top:
        span = int(cell.get("colspan", 1))
        if span > 1:
            groups.append((plain(cell), [plain(s) for s in subs[sub_index:sub_index + span]]))
            sub_index += span
        else:
            groups.append((plain(cell), [""]))   # rowspan=2, 통계 이름이 따로 없는 지표
    if not groups:
        raise ValueError("reference-similarity 표의 헤더에서 지표 열을 찾지 못했습니다.")
    n_values = sum(len(stats) for _, stats in groups)

    methods, values_by_method = [], []
    for row in table.select("tbody > tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) != n_values + 1:
            continue
        methods.append(plain(cells[0]))
        values_by_method.append([
            format_similarity_two_decimal_places(cell_to_latex(cell, keep_badges=False))
            if REFERENCE_SIMILARITY_TWO_DECIMALS
            else cell_to_latex(cell, keep_badges=False)
            for cell in cells[1:]
        ])
    if not methods:
        raise ValueError(
            f"값 {n_values}개짜리 method 행이 없습니다. "
            "fragment를 build_reference_similarity.py로 다시 만들어 주세요."
        )

    pad = "        " if wrap else "    "        # \begin{tabular} 들여쓰기
    line = pad + "    "                        # 그 안의 행
    body = [
        r"\begin{wraptable}{r}{0.55\textwidth}" if wrap else r"\begin{table}[t]",
        r"    \centering",
        r"    \caption{",
        "        " + REFERENCE_SIMILARITY_CAPTION,
        r"    }",
        rf"    \label{{{REFERENCE_SIMILARITY_LABEL}}}",
    ]
    if wrap:
        body.append(r"    \resizebox{.98\linewidth}{!}{%")
    body += [
        pad + rf"\begin{{tabular}}{{@{{}}ll{'c' * len(methods)}@{{}}}}",
        line + r"\toprule",
        line + " & ".join([r"\multicolumn{2}{@{}l}{\textbf{Method}}"]
                          + [rf"\textbf{{{method}}}" for method in methods]) + r" \\",
        line + r"\midrule",
    ]
    value_index = 0
    for group_index, (name, stats) in enumerate(groups):
        if group_index:
            body.append(line + r"\addlinespace")
        single = len(stats) == 1 and not stats[0]
        for stat_index, stat in enumerate(stats):
            values = [row[value_index] for row in values_by_method]
            value_index += 1
            if single:
                head = (r"\multicolumn{2}{@{}l}{\shortstack[l]{"
                        + r"\\ ".join(name.split(" ", 1)) + "}}")
            elif stat_index == 0:
                head = rf"\multirow{{{len(stats)}}}{{*}}{{{name}}} & {stat}"
            else:
                head = f" & {stat}"
            body.append(line + " & ".join([head, *values]) + r" \\")
    body += [line + r"\bottomrule", pad + r"\end{tabular}"]
    if wrap:
        body.append(r"    }")
    body.append(r"\end{wraptable}" if wrap else r"\end{table}")
    return "\n".join(body)


def drug_design_vina_to_latex(table: Tag) -> str:
    """Table 1 of results_drug_design.html -> LaTeX (task-2 de novo drug design).

    Fixed caption; VALUES pulled live from the HTML. Column order: Method, n, then
    Vina evaluation (Score/Min/Dock as Avg./Med. + High aff.) and Sample quality
    (QED/SA/Div as Avg./Med. + Heavy atoms). 'Method' sits in the bottom header row
    (no multirow); 'n' is pulled to the front, outside both groups; High aff. and Heavy
    atoms are vertically centred with \\multirow{2}. Best/second emphasis is applied
    automatically per metric column among the GENERATED methods (Reference excluded;
    Score/Min/Dock lower-is-better, the rest higher; Heavy atoms and n get none). Only
    one Ours row is kept (the CDG result); \\oursC is an empty coords-ablation placeholder.
    """
    NAME = {
        "Reference ligand": "Reference",
        "VoxBind σ=0.9": r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
        "Ours · v1": r"\;+ \ours{}",
        "Ours · v2": r"\;+ \ours{}",
        "Ours · v3": r"\;+ \ours{}",
    }
    LOWER  = {0, 1, 2, 3, 4, 5}          # Score/Min/Dock (avg, med) -> lower better
    HIGHER = {6, 7, 8, 9, 10, 11, 12}    # High aff. + QED/SA/Div    -> higher better
    # 13 = Heavy atoms, 14 = n : no emphasis
    REORDER = [14] + list(range(14))     # emit n first, then Score..Heavy
    OURS_BG = r"\rowcolor{gray!15}"      # light bg on our rows (needs \usepackage[table]{xcolor})
    QUAL_2DP = {7, 8, 9, 10, 11, 12}     # QED/SA/Div (avg, med) -> 2 decimals

    def clean(cell: Tag) -> str:
        t = cell.get_text(" ", strip=True).replace("−", "-").replace("—", "-").replace("–", "-")
        return t if t else "-"

    def raw_name(cell: Tag) -> str:
        c = cell.__copy__()
        for junk in c.select(".tag, .sub, .nsub, span[style]"):
            junk.decompose()
        return c.get_text(" ", strip=True)

    def num(v):
        try:
            return float(v.replace(",", "").replace("−", "-"))
        except Exception:
            return None

    collected, seen_ours = [], False
    for r in table.select("tbody > tr"):
        cells = r.find_all(["th", "td"], recursive=False)
        if len(cells) == 1:
            collected.append({"kind": "divider"}); continue
        if len(cells) != 16:
            continue
        name = raw_name(cells[0])
        vals = [clean(c) for c in cells[1:]]
        hspan = cells[14].find("span")
        heavy_med = hspan.get("data-med") if hspan and hspan.get("data-med") else ""
        if any(v == "TBA" for v in vals):
            continue
        if name.startswith("Ours"):
            if seen_ours:
                continue
            seen_ours = True
        collected.append({"kind": "data", "name": name, "vals": vals, "heavy_med": heavy_med})
    for i, c in enumerate(collected):
        c["rid"] = i

    gen = [c for c in collected if c.get("kind") == "data" and c["name"] != "Reference ligand"]
    emph = {}
    for col in range(15):
        if col not in LOWER and col not in HIGHER:
            continue
        pairs = [(c["rid"], num(c["vals"][col])) for c in gen if num(c["vals"][col]) is not None]
        if not pairs:
            continue
        ordered = sorted(pairs, key=lambda x: x[1], reverse=(col in HIGHER))
        best = ordered[0][1]
        second = next((v for _, v in ordered if v != best), None)
        for rid, v in pairs:
            if v == best:
                emph[(rid, col)] = "bold"
            elif second is not None and v == second:
                emph[(rid, col)] = "under"

    lines = []
    for c in collected:
        if c.get("kind") == "divider":
            if lines and lines[-1] != r"\midrule":
                lines.append(r"\midrule")
            continue
        name, rid = c["name"], c["rid"]
        disp = NAME.get(name, escape_latex_text(name))
        cells_out = []
        for col in REORDER:
            v = c["vals"][col]
            if col in QUAL_2DP:
                f = num(v)
                if f is not None:
                    v = "%.2f" % f
            e = emph.get((rid, col))
            if e == "bold":
                v = r"\textbf{%s}" % v
            elif e == "under":
                v = r"\underline{%s}" % v
            cells_out.append(v)
        cells_out.append(c.get("heavy_med") or "-")   # # atoms / mol : Med from HTML data-med (baselines only)
        row = " & ".join([disp, *cells_out]) + r" \\"
        if name.startswith("Ours"):
            row = OURS_BG + " " + row
        lines.append(row)
        if name == "Reference ligand":
            lines.append(r"\midrule")
        if name == "VoxBind σ=0.9":
            lines.append(OURS_BG + r" \;+ \oursC{} & \\")

    caption = (
        r"\textbf{Pocket-conditioned ligand generation results on the CrossDocked2020 benchmark} "
        r"for 79 test pockets with experimental density maps. "
        r"High aff. denotes the percentage of the generated molecules with a better Vina "
        r"docking score than the reference ligand, and $n$ denotes the number of generated molecules."
    )
    todo = r"\todo{$\ddagger$: sampling ongoing}"
    header = [
        r"\toprule",
        r"& & \multicolumn{7}{c}{\textbf{Vina evaluation}} & \multicolumn{8}{c}{\textbf{Sample quality}} \\",
        r"\cmidrule(lr){3-9}\cmidrule(lr){10-17}",
        r"& & \multicolumn{2}{c}{Score $\downarrow$} & \multicolumn{2}{c}{Min $\downarrow$} & \multicolumn{2}{c}{Dock $\downarrow$} & \multirow{2}{*}[-3.5pt]{\shortstack{High\,$\uparrow$ \\ aff.\,\%}} & \multicolumn{2}{c}{QED $\uparrow$} & \multicolumn{2}{c}{SA $\uparrow$} & \multicolumn{2}{c}{Div. $\uparrow$} & \multicolumn{2}{c}{\shortstack{\# atoms / mol}} \\",
        r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}\cmidrule(lr){10-11}\cmidrule(lr){12-13}\cmidrule(lr){14-15}\cmidrule(lr){16-17}",
        r"\textbf{Method} & $n$ & Avg. & Med. & Avg. & Med. & Avg. & Med. & & Avg. & Med. & Avg. & Med. & Avg. & Med. & Avg. & Med. \\",
        r"\midrule",
    ]

    def ind(rows, level):
        return [INDENT * level + r for r in rows]

    out = (
        [r"\begin{table}[t]"]
        + ind([r"\centering", r"\caption{"], 1)
        + ind([caption, todo], 2)
        + ind([r"}", r"\label{tab:result-drug-design}", r"\resizebox{.98\textwidth}{!}{%"], 1)
        + ind([r"\begin{tabular}{lcccccccccccccccc}"], 2)
        + ind([*header, *lines, r"\bottomrule"], 3)
        + ind([r"\end{tabular}"], 2)
        + ind([r"}"], 1)
        + [r"\end{table}"]
    )
    return "\n".join(out)


## Load report + list tables

In [3]:
dd_path = resolve_html_path(DRUG_DESIGN_HTML_PATH)
dd_soup = BeautifulSoup(dd_path.read_text(encoding="utf-8"), "html.parser")
def _dd_title(tbl):
    t = tbl.find_previous("p", class_="table-title")
    return t.get_text(" ", strip=True) if t else ""
dd_tables = dd_soup.find_all("table")
print(f"Source: {dd_path}  ·  {len(dd_tables)} tables")
for i, t in enumerate(dd_tables):
    print(f"  [{i}] {_dd_title(t)}")


Source: /home/shpark/prj-denovo/VoxBind/results/reports/results_drug_design.html  ·  14 tables
  [0] Run progress  ·  sampling → docking → PoseCheck → PoseBusters
  [1] Table 1  ·  Vina affinity and sample quality — 79 pockets, whole receptor
  [2] Table 2  ·  Vina Dock by generated-ligand size — 78 pockets
  [3] Table 3  ·  Where each population sits on the size axis
  [4] Table 4  ·  Pose quality — 78 pockets
  [5] Table 5  ·  PoseBusters — which checks actually fail
  [6] Table 6  ·  Strain energy — the distribution, not just the median
  [7] Table 7  ·  Protein–ligand interaction profile
  [8] Table 8  ·  Does pose quality trade against docking score?
  [9] Table A1  ·  Paired comparison against vanilla VoxBind — 78 pockets
  [10] Table A2  ·  Cropped vs whole receptor — 78 pockets, same molecules
  [11] Table A3  ·  TargetDiff — pocket-set sensitivity (our run)
  [12] Table A4  ·  Published values — VoxBind paper, Table 1 (100 pockets)
  [13] Table A5  ·  Cropped-receptor protocol

## Vina affinity + sample quality

In [4]:
# de novo Vina affinity + sample quality
vina = [t for t in dd_tables if "Vina affinity" in _dd_title(t)]
print(drug_design_vina_to_latex(vina[0]))


\begin{table}[t]
    \centering
    \caption{
        \textbf{Pocket-conditioned ligand generation results on the CrossDocked2020 benchmark} for 79 test pockets with experimental density maps. High aff. denotes the percentage of the generated molecules with a better Vina docking score than the reference ligand, and $n$ denotes the number of generated molecules.
        \todo{$\ddagger$: sampling ongoing}
    }
    \label{tab:result-drug-design}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{lcccccccccccccccc}
            \toprule
            & & \multicolumn{7}{c}{\textbf{Vina evaluation}} & \multicolumn{8}{c}{\textbf{Sample quality}} \\
            \cmidrule(lr){3-9}\cmidrule(lr){10-17}
            & & \multicolumn{2}{c}{Score $\downarrow$} & \multicolumn{2}{c}{Min $\downarrow$} & \multicolumn{2}{c}{Dock $\downarrow$} & \multirow{2}{*}[-3.5pt]{\shortstack{High\,$\uparrow$ \\ aff.\,\%}} & \multicolumn{2}{c}{QED $\uparrow$} & \multicolumn{2}{c}{SA $\uparrow$} & \multicolumn{

## Pose quality / PoseBusters / strain / interactions

In [5]:
# PoseCheck / PoseBusters / strain / interaction tables (generic renderer)
POSE_TITLES = ["Pose quality", "PoseBusters", "Strain energy", "interaction profile"]
for key in POSE_TITLES:
    hit = [t for t in dd_tables if key.lower() in _dd_title(t).lower()]
    if not hit:
        print(f"% (no table titled ~{key!r})\n"); continue
    print(f"% ── {_dd_title(hit[0])} ──")
    print(table_to_latex(hit[0], 99, keep_badges=KEEP_BADGES, resize_wide=RESIZE_WIDE))
    print()


% ── Table 4  ·  Pose quality — 78 pockets ──
% Requires: \usepackage{booktabs,multirow,graphicx}
\begin{table}[t]
    \centering
    \caption{
        Pose quality --- 78 pockets
    }
    \label{tab:results-4}
    \begin{tabular}{@{}lllllll@{}}
        \toprule
        \multirow{2}{*}{\textbf{Method}} & \multicolumn{2}{c}{\textbf{Strain energy $\downarrow$}} & \multicolumn{2}{c}{\textbf{Steric clashes $\downarrow$}} & \multirow{2}{*}{\shortstack[c]{\textbf{PB-valid} \\ \textbf{\% $\uparrow$}}} & \multirow{2}{*}{\textbf{n mol.}} \\
        & \textbf{Median} & \textbf{IQR} & \textbf{Mean} & \textbf{Median} &  &  \\
        \midrule
        \multicolumn{7}{c}{Reference \& prior methods $\cdot$ our runs on the 78-pocket set} \\
        \shortstack[l]{Reference ligand \\ crystal pose $\cdot$ 1 per pocket} & 33.9 & 10.7 -- 68.3 & 6.82 & 5.0 & 95.2 n=42 & 78 \\
        AR\,\textsuperscript{\scriptsize pending} & TBA & TBA & TBA & TBA & TBA & --- \\
        Pocket2Mol\,\textsuperscript{\scri

## Reference-ligand similarity

In [6]:
rs_path = resolve_html_path(REFERENCE_SIMILARITY_HTML_PATH)
rs_table = BeautifulSoup(rs_path.read_text(encoding="utf-8"), "html.parser").find("table")
print(reference_similarity_to_latex(rs_table))
print("\n% ── wrapfig variant ──")
print(reference_similarity_to_latex(rs_table, wrap=True))


\begin{table}[t]
    \centering
    \caption{
        \textbf{Reference-ligand similarity} on the CrossDocked benchmark. Metrics are averaged over pockets.
    }
    \label{tab:result-drug-reference-similarity}
    \begin{tabular}{@{}llccccccccc@{}}
        \toprule
        \multicolumn{2}{@{}l}{\textbf{Method}} & \textbf{AR} & \textbf{Pocket2Mol} & \textbf{DiffSBDD} & \textbf{DecompDiff} & \textbf{TargetDiff} & \textbf{FuncBind} & \textbf{VoxBind\textsubscript{\scriptsize $\sigma$=0.9}} & \textbf{VoxBind\textsubscript{\scriptsize $\sigma$=1.0}} & \textbf{Ours\textsubscript{\scriptsize v1}} \\
        \midrule
        \multirow{2}{*}{ECFP4} & Avg. & 0.100 & 0.097 & 0.089 & 0.152 & 0.091 & 0.118 & 0.099 & 0.103 & 0.099 \\
         & Med. & 0.096 & 0.092 & 0.085 & 0.137 & 0.087 & 0.110 & 0.092 & 0.099 & 0.094 \\
        \addlinespace
        \multicolumn{2}{@{}l}{\shortstack[l]{Scaffold\\ match}} & 1.04\% & 1.12\% & 0.55\% & 2.17\% & 0.62\% & 1.30\% & 0.96\% & 1.18\% & 0.81\% \\
        